# Notebook de documentacion, tratamiento datos y entrenamiento


## Equipo
- Alumno 1 : Maximiliano Romano 
- Alumno 2 : Facundo Molina

In [ ]:
## TO-DO utiliza esta notebook para documentar, entrenar y probar el modelo.

**LIBRERÍAS**

In [18]:
import cv2
import os
from pathlib import Path
from sklearn.datasets import fetch_lfw_people
import numpy as np
import insightface
from PIL import Image
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
from facenet_pytorch import InceptionResnetV1
import torch
from torchvision import transforms
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
import sys


**DATASET**

In [19]:
# DESCARGA DE DATASET LFW, RECORTE DE 20 PRINCIPALES Y GUARDADO EN CARPETA
lfw_people = fetch_lfw_people(min_faces_per_person=40, resize=1.0)

unique, counts = np.unique(lfw_people.target, return_counts=True)

top_20 = unique[np.argsort(counts)[-20:]]

mask = np.isin(lfw_people.target, top_20)
X = lfw_people.images[mask]
y = lfw_people.target[mask]


new_labels = {old: new for new, old in enumerate(top_20)}
y = np.array([new_labels[label] for label in y])

target_names = lfw_people.target_names[top_20]

base_path = Path(r"C:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp1\tuia-face-recognition-app\data\processed")


for name in target_names:
    person_path = base_path / name.replace(" ", "_")
    person_path.mkdir(parents=True, exist_ok=True)

for i, (img, label_idx) in enumerate(zip(X, y)):

    name = target_names[label_idx].replace(" ", "_")

    img_to_save = (img * 255).astype(np.uint8)

    img_to_save = cv2.cvtColor(img_to_save, cv2.COLOR_GRAY2BGR)

    file_name = f"{name}_{i:04d}.jpg"
    file_path = base_path / name / file_name

    cv2.imwrite(str(file_path), img_to_save)

**DETECCIÓN Y ALINEACIÓN**

In [20]:
os.makedirs("models", exist_ok=True)

model = InceptionResnetV1(pretrained='vggface2', classify=False).eval()

torch.save(model, "models/inceptionresnetv1_finetuned.pth")

In [21]:

root_path = Path(os.getcwd())
sys.path.append(str(root_path / "src"))

from src.lib.schemas import AlignedFace, EmbeddingRecord, InsertRequest
from src.lib.services.face_service import FaceService


model_path=Path("models/inceptionresnetv1_finetuned.pth")
checkpoint_path = Path("models/inceptionresnetv1_finetuned.pth")

if checkpoint_path.exists():
    service = FaceService(
        store=None, 
        similarity_metric="cosine",
        similarity_threshold=0.6,
        face_size=160,
        model_path=checkpoint_path)
    print("Servicio instanciado con éxito.")
else:
    print(f"Error: No se encontró el archivo en {checkpoint_path}")



def procesar_alineacion_adecuada(image_bgr, box_esperado):

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    from PIL import Image
    img_pil = Image.fromarray(image_rgb)


    boxes, probs, landmarks = service.detector.detect(img_pil, landmarks=True)
    faces_tensor = service.detector(img_pil)

    x1, y1, x2, y2 = box_esperado
    best_idx = 0


    if boxes is not None and len(boxes) > 1:
        min_dist = float("inf")
        for i, b in enumerate(boxes):

            dist = abs(int(b[0]) - x1) + abs(int(b[1]) - y1)
            if dist < min_dist:
                min_dist = dist
                best_idx = i

    kps = landmarks[best_idx] if landmarks is not None else None


    if faces_tensor is None:

        xlc, ylc, x2c, y2c = np.clip([x1, y1, x2, y2], 0, [image_bgr.shape[1], image_bgr.shape[0], image_bgr.shape[1], image_bgr.shape[0]])
        crop = cv2.resize(image_bgr[int(ylc):int(y2c), int(xlc):int(x2c)], (service.face_size, service.face_size))
        return AlignedFace(bbox=np.array(box_esperado), keypoints=kps, image=crop, embedding=None)

    t = faces_tensor[best_idx] if faces_tensor.ndim == 4 else faces_tensor
    

    face_np = ((t.permute(1, 2, 0).cpu().detach().numpy() + 1) * 127.5).clip(0, 255).astype(np.uint8)
    face_bgr = cv2.cvtColor(face_np, cv2.COLOR_RGB2BGR)
    
    bbox_final = np.array(boxes[best_idx]) if boxes is not None else np.array(box_esperado)
    
    return AlignedFace(bbox=bbox_final, keypoints=kps, image=face_bgr, embedding=None)

Servicio instanciado con éxito.


**ARQUITECTURA(CON INCEPTIONRESNETV1)**

In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

modelo = InceptionResnetV1(pretrained="vggface2").eval().to(device)

print(f"Parámetros: {sum(p.numel() for p in modelo.parameters()):,}")

dummy = torch.randn(1, 3, 160, 160).to(device)

with torch.no_grad():
    emb = modelo(dummy)
    
print(f"Embedding shape: {emb.shape}  norma L2: {emb.norm().item():.4f} (esperado ~ 1.0)")

Parámetros: 27,910,327
Embedding shape: torch.Size([1, 512])  norma L2: 1.0000 (esperado ~ 1.0)


**EXTRACCIÓN DE EMBEDDINGS**

In [ ]:
input_root = Path(r"C:\Users\fjm25\Desktop\Facundo\TUIA\CV\tp1\tuia-face-recognition-app\src\data\processed")
output_dir = Path(r"data\aligned_faces") 
output_dir.mkdir(parents=True, exist_ok=True)

image_files = list(input_root.rglob("*.jpg"))

if not image_files:
    print(f"No se encontraron imágenes en  {input_root}")
else:
    print(f"Se encontraron {len(image_files)} imágenes en total")

    for img_path in image_files:

        k = img_path.parent.name 
        
        try:
            img_bgr = cv2.imread(str(img_path))
            if img_bgr is None: continue

            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            img_pil = Image.fromarray(img_rgb)

            boxes, _ = service.detector.detect(img_pil)

            if boxes is not None:
                for i, box in enumerate(boxes):
     
                    box_int = tuple(map(int, map(round, box[:4])))
                    aligned_obj = service.align_face(img_bgr, box_int)
                        
                    if aligned_obj is None: 
                        continue

                    face_rgb = cv2.cvtColor(aligned_obj.image, cv2.COLOR_BGR2RGB)
                    face_pil = Image.fromarray(face_rgb)
                    face_tensor = preprocess(face_pil).unsqueeze(0).to(device)


                    with torch.no_grad():
                        embedding_tensor = modelo(face_tensor)
                        embedding_flat = embedding_tensor.squeeze().cpu().numpy()

                    face_id = f"{k}_{img_path.stem}_{i}"
                    embeddings_db[face_id] = embedding_flat
                    

                    face_pil.save(output_dir / f"{face_id}.jpg")
                
                print(f"✅ Procesado: {k} ({img_path.name})")

        except Exception as e:
            print(f"Error procesando {img_path}: {e}")

    print(f"\n terminado")
    print(f"Total de caras en database: {len(embeddings_db)}")

Se encontraron 3359 imágenes en total. Iniciando proceso...
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0053.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0059.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0060.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0067.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0113.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0125.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0126.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0127.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0140.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0142.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0160.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0170.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0180.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0186.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0189.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0190.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0193.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_0208.jpg)
✅ Procesado: Ariel_Sharon (Ariel_Sharon_

**VERIFICACIÓN**

In [ ]:
def verificar(emb1, emb2, umbral=0.6):
    sim = float(np.dot(emb1, emb2))
    return {"similitud": round(sim, 4), "misma_persona": sim >= umbral}

emb_array = np.array(list(embeddings_db.values()))
emb_labels = list(embeddings_db.keys())


sim_matrix = emb_array @ emb_array.T


fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=0, vmax=1)

plt.colorbar(im, ax=ax, label="Cosine Similarity")


ticks = range(len(emb_labels))
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(emb_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(emb_labels, fontsize=8)


for i in range(len(emb_labels)):
    for j in range(len(emb_labels)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", 
                ha="center", va="center", color="black", fontsize=7)

ax.set_title("Matriz de Similitud entre Imágenes Procesadas")
plt.tight_layout()
plt.show()

**EVALUACIÓN**

**FINE-TUNING**

In [ ]:
own_tensors = []
own_labels_raw = []

face_rgb = cv2.cvtColor(aligned_obj.image, cv2.COLOR_BGR2RGB)
face_pil = Image.fromarray(face_rgb)

face_t = preprocess(face_pil) 

own_tensors.append(face_t)
own_labels_raw.append(persona_name)


le_ft = LabelEncoder()
y_all = torch.tensor(le_ft.fit_transform(own_labels_raw), dtype=torch.long)
X_all = torch.stack(own_tensors)

loader = DataLoader(TensorDataset(X_all, y_all), batch_size=8, shuffle=True)
print(f"Fine-tuning: {len(own_tensors)} imágenes | {len(le_ft.classes_)} clases")


for p in modelo.parameters():
    p.requires_grad = False


for p in modelo.last_linear.parameters():
    p.requires_grad = True
for p in modelo.last_bn.parameters():
    p.requires_grad = True

num_clases = len(le_ft.classes_)
head = nn.Linear(512, num_clases).to(device)

optimizer = torch.optim.Adam(
    list(modelo.last_linear.parameters()) + 
    list(modelo.last_bn.parameters()) + 
    list(head.parameters()), 
    lr=1e-4
)
criterion = nn.CrossEntropyLoss()


modelo.train()
head.train()

for epoch in range(10):
    total_loss = 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        

        embeddings = modelo(batch_x)

        outputs = head(embeddings)
        
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/10 - Loss: {total_loss/len(loader):.4f}")



**GUARDADO**

In [ ]:
# from src.lib.schemas import AlignedFace, EmbeddingRecord, InsertRequest